# AMD Scalper Training Notebook
## 5-Minute Session Manipulation Reversal — GPU Training

**AMD Scalper** identifies high-probability scalping entries during the New York session using the **Accumulation–Manipulation–Distribution (AMD)** framework:

- **Accumulation (Asian session)**: Tight range with liquidity building on both sides
- **Manipulation (London session)**: Aggressive sweep of the Asian range (stop hunt)
- **Distribution (New York session)**: Reversal entry via one of 3 models:
  1. **S&R Model** — Horizontal flip zones + engulfing patterns
  2. **S&D Model** — Demand/supply zones + Fair Value Gaps
  3. **ORB Model** — NY opening range breakout + retracement

**Architecture**: Multi-TF (5m/15m/1h/4h) with AMD-specific encoders, session-aware fusion, gated entry model ensemble.

**Data**: Resamples 1-minute raw data to 5-minute bars, reuses existing 15m/1h/4h processed parquets. AMD features (Asian range, London sweep, engulfing, FVG, S&R zones, ORB) computed on the fly.

**OANDA Account**: `101-001-30902818-003` (separate from MTF `-001` and WaveFollower `-002`)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 1: Imports & GPU Check (with performance flags)
# ═══════════════════════════════════════════════════════════════
import sys, os, time, json, hashlib
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split, ConcatDataset, Dataset
from torch.amp import autocast, GradScaler

# Ensure wavetrader package is importable
sys.path.insert(0, str(Path.cwd()))

# GPU check
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch version : {torch.__version__}")
print(f"Device          : {device}")
if device.type == "cuda":
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"VRAM            : {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
    print(f"CUDA version    : {torch.version.cuda}")
    # Performance flags
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    print(f"cuDNN benchmark : enabled")
    print(f"TF32            : enabled")
else:
    print("WARNING: No GPU detected — training will use CPU (slow).")

## 2. Configuration & Data Paths

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 2: Configuration — AMD Scalper hyperparameters
# ═══════════════════════════════════════════════════════════════
from wavetrader.amd_scalper import AMDScalper
from wavetrader.config import AMDScalperConfig

config = AMDScalperConfig(
    # --- Training hyperparams (GPU-optimized) ---
    learning_rate=2e-4,  # Scaled with batch size (linear scaling rule)
    batch_size=640,      # Max for T4 16GB; reduce if OOM
    epochs=50,
    dropout=0.20,
)

# Data paths
PROCESSED_DIR = Path("processed_data")
RAW_DATA_DIR = Path("data")
CHECKPOINT_DIR = Path("checkpoints/amd_scalper")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Pairs to train on (focus on JPY pairs — strongest session dynamics)
PAIRS = ["GBPJPY", "EURJPY", "GBPUSD"]
PAIR_NAMES = {"GBPJPY": "GBP/JPY", "EURJPY": "EUR/JPY", "GBPUSD": "GBP/USD"}

# AMD-specific pip sizes for JPY and non-JPY pairs
PIP_SIZES = {"GBPJPY": 0.01, "EURJPY": 0.01, "GBPUSD": 0.0001}

# Scalping: shorter lookahead (6 bars × 5min = 30min) and tighter threshold
LOOKAHEAD = 6
THRESHOLD_PIPS = 15.0

print(f"Config: {config.timeframes}, lookbacks={config.lookbacks}")
print(f"Entry TF: {config.entry_timeframe}")
print(f"Batch size: {config.batch_size}, LR: {config.learning_rate}, Epochs: {config.epochs}")
print(f"Lookahead: {LOOKAHEAD} bars ({LOOKAHEAD * 5} min), Threshold: {THRESHOLD_PIPS} pips")
print(f"Pairs: {list(PAIR_NAMES.values())}")
print(f"Checkpoint dir: {CHECKPOINT_DIR}")

## 3. Load & Resample Data

AMD Scalper uses **5-minute** as its entry timeframe. Since we only have 1-minute raw CSVs and 15m+ processed parquets, we:
1. Load 1-minute raw Dukascopy CSVs
2. Resample to 5-minute OHLCV bars
3. Load existing 15m/1h/4h from processed parquets
4. Split all timeframes chronologically into train/val/test

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 3: Load 1-minute data and resample to 5-minute bars
# ═══════════════════════════════════════════════════════════════
from wavetrader.data import load_forex_data, _normalise_df, _resample_ohlcv

def load_5min_from_1min(pair_tag, raw_dir):
    """Load 1-minute CSVs (ask+bid), compute mid-price, resample to 5m."""
    pair_name = PAIR_NAMES[pair_tag]

    # Try loading ask and bid separately for mid-price
    ask_files = sorted(raw_dir.glob(f"{pair_tag}*1 Min*Ask*.csv"))
    bid_files = sorted(raw_dir.glob(f"{pair_tag}*1 Min*Bid*.csv"))

    if ask_files and bid_files:
        print(f"  Loading ask+bid 1min for {pair_tag}...")
        ask_df = _normalise_df(pd.read_csv(ask_files[0]))
        bid_df = _normalise_df(pd.read_csv(bid_files[0]))

        # Align on date, compute mid-price
        merged = ask_df.merge(bid_df, on="date", suffixes=("_ask", "_bid"))
        mid = pd.DataFrame({
            "date": merged["date"],
            "open": (merged["open_ask"] + merged["open_bid"]) / 2,
            "high": (merged["high_ask"] + merged["high_bid"]) / 2,
            "low": (merged["low_ask"] + merged["low_bid"]) / 2,
            "close": (merged["close_ask"] + merged["close_bid"]) / 2,
            "volume": merged.get("volume_ask", merged.get("volume_bid", 0)),
        })
    else:
        # Fallback: load single 1min file
        print(f"  Loading single 1min file for {pair_tag}...")
        one_min_files = sorted(raw_dir.glob(f"{pair_tag}*1 Min*.csv"))
        if not one_min_files:
            print(f"  WARNING: No 1min data found for {pair_tag}")
            return None
        mid = _normalise_df(pd.read_csv(one_min_files[0]))

    # Resample to 5 minutes
    df_5m = _resample_ohlcv(mid, "5min")
    print(f"  {pair_tag} 1min→5min: {len(mid):,} → {len(df_5m):,} bars")
    return df_5m


def load_higher_tf(pair_tag, tf, processed_dir, raw_dir):
    """Load higher TF from processed parquets (or raw CSV fallback)."""
    tf_short = tf.replace("min", "m")

    # Try each split directory (we'll merge them back)
    frames = []
    for split in ["train", "val", "test"]:
        path = processed_dir / split / f"{pair_tag}_{tf_short}.parquet"
        if path.exists():
            df = pd.read_parquet(path)
            df = _normalise_df(df)
            frames.append(df)

    if frames:
        combined = pd.concat(frames).drop_duplicates(subset=["date"]).sort_values("date").reset_index(drop=True)
        print(f"  {pair_tag} {tf}: {len(combined):,} bars (from processed parquets)")
        return combined

    # Fallback to raw
    pair_name = PAIR_NAMES[pair_tag]
    df = load_forex_data(pair_name, tf, data_dir=str(raw_dir))
    if df is not None and not df.empty:
        print(f"  {pair_tag} {tf}: {len(df):,} bars (from raw CSV)")
        return df

    print(f"  WARNING: No data for {pair_tag} {tf}")
    return None


# Load all data
all_pair_data = {}
for pair_tag in PAIRS:
    print(f"\n{'='*50}")
    print(f"Loading {pair_tag}:")
    dfs = {}

    # 5min from 1min resampling
    df_5m = load_5min_from_1min(pair_tag, RAW_DATA_DIR)
    if df_5m is not None:
        dfs["5min"] = df_5m

    # Higher TFs from processed parquets
    for tf in ["15min", "1h", "4h"]:
        df_tf = load_higher_tf(pair_tag, tf, PROCESSED_DIR, RAW_DATA_DIR)
        if df_tf is not None:
            dfs[tf] = df_tf

    if len(dfs) == len(config.timeframes):
        all_pair_data[pair_tag] = dfs
        print(f"  ✓ All {len(config.timeframes)} timeframes loaded")
    else:
        print(f"  ✗ Only {len(dfs)}/{len(config.timeframes)} TFs — skipping")

print(f"\n{'='*50}")
print(f"Pairs ready: {list(all_pair_data.keys())}")

## 4. Chronological Split (Train / Val / Test)

5-minute scalping models require strict chronological splits to avoid look-ahead bias. We use ~70/15/15 by date.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 4: Chronological split — 70/15/15 by date
# ═══════════════════════════════════════════════════════════════

def chrono_split_mtf(pair_dfs, train_frac=0.70, val_frac=0.15):
    """Split all timeframes for a pair at consistent date boundaries."""
    # Use entry TF (5min) dates as reference
    entry_tf = config.entry_timeframe
    entry_df = pair_dfs[entry_tf]
    n = len(entry_df)
    train_end = int(n * train_frac)
    val_end = int(n * (train_frac + val_frac))

    train_date = entry_df.iloc[train_end]["date"]
    val_date = entry_df.iloc[val_end]["date"]

    splits = {"train": {}, "val": {}, "test": {}}
    for tf, df in pair_dfs.items():
        mask_train = df["date"] < train_date
        mask_val = (df["date"] >= train_date) & (df["date"] < val_date)
        mask_test = df["date"] >= val_date

        splits["train"][tf] = df[mask_train].reset_index(drop=True)
        splits["val"][tf] = df[mask_val].reset_index(drop=True)
        splits["test"][tf] = df[mask_test].reset_index(drop=True)

    return splits


# Split each pair
split_data = {}  # {pair_tag: {"train": {tf: df}, "val": {...}, "test": {...}}}
for pair_tag, dfs in all_pair_data.items():
    splits = chrono_split_mtf(dfs)
    split_data[pair_tag] = splits

    entry_tf = config.entry_timeframe
    print(f"{pair_tag}:")
    for split_name in ["train", "val", "test"]:
        n_entry = len(splits[split_name][entry_tf])
        date_range = ""
        if n_entry > 0:
            d0 = splits[split_name][entry_tf].iloc[0]["date"]
            d1 = splits[split_name][entry_tf].iloc[-1]["date"]
            date_range = f" ({d0:%Y-%m-%d} → {d1:%Y-%m-%d})"
        print(f"  {split_name:5s}: {n_entry:>8,} 5min bars{date_range}")

## 5. Verify AMD Feature Engineering

Quick check: ensure `build_amd_features()` produces all 23 AMD features correctly on 5-minute data.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 5: Validate AMD feature engineering on sample data
# ═══════════════════════════════════════════════════════════════
from wavetrader.dataset import prepare_features
from wavetrader.amd_features import build_amd_features
from wavetrader.amd_dataset import AMD_FEATURE_COLS

# Take a sample
sample_pair = list(split_data.keys())[0]
sample_split = split_data[sample_pair]["train"]
sample_df = sample_split["5min"].copy()

# Run standard prepare + AMD features
prepared = prepare_features(sample_df, lookahead=LOOKAHEAD, pair=PAIR_NAMES[sample_pair])
amd_enriched = build_amd_features(prepared)

print(f"Sample: {sample_pair} 5min train — {len(amd_enriched):,} rows")
print(f"Total columns: {len(amd_enriched.columns)}")

# Check standard features
STANDARD_FEATURES = {
    "ohlcv":     ["open_norm", "high_norm", "low_norm", "close_norm", "volume_norm"],
    "structure": [f"structure_{i}" for i in range(8)],
    "rsi":       ["rsi_norm", "rsi_delta_norm", "rsi_accel_norm"],
    "volume":    ["volume_norm", "volume_ratio", "volume_delta"],
    "regime":    ["session_tokyo", "session_london", "session_newyork", "atr_pct"],
}

print("\nStandard features:")
for group, cols in STANDARD_FEATURES.items():
    present = all(c in amd_enriched.columns for c in cols)
    print(f"  {group:12s}: {'OK' if present else 'MISSING'} ({len(cols)} features)")

# Check AMD features
print(f"\nAMD features ({len(AMD_FEATURE_COLS)} expected):")
amd_present = [c for c in AMD_FEATURE_COLS if c in amd_enriched.columns]
amd_missing = [c for c in AMD_FEATURE_COLS if c not in amd_enriched.columns]
print(f"  Found: {len(amd_present)}/{len(AMD_FEATURE_COLS)}")
if amd_missing:
    print(f"  Missing: {amd_missing}")

# Check AMD phase labels
if "amd_phase" in amd_enriched.columns:
    phase_dist = amd_enriched["amd_phase"].value_counts().sort_index()
    phase_names = {0: "ACCUM", 1: "MANIP", 2: "DIST", 3: "INVALID"}
    print(f"\nAMD Phase distribution:")
    for idx, count in phase_dist.items():
        pct = count / len(amd_enriched) * 100
        print(f"  {phase_names.get(idx, '?'):8s} ({idx}): {count:>8,} ({pct:.1f}%)")

# Sample some non-zero AMD features
print(f"\nAMD feature non-zero rates:")
for col in AMD_FEATURE_COLS[:10]:
    nz = (amd_enriched[col] != 0).sum()
    print(f"  {col:25s}: {nz:>8,} / {len(amd_enriched):,} ({nz/len(amd_enriched)*100:.1f}%)")

## 6. Build Datasets & DataLoaders

AMD Scalper uses session-aware features, so each pair's data is processed separately with `build_amd_features()`. We then concatenate all pairs and pre-cache into contiguous tensors for maximum training speed.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 6: Pre-cached AMD dataset + DataLoaders (critical for speed)
# ═══════════════════════════════════════════════════════════════
from wavetrader.amd_dataset import AMDForexDataset, amd_collate_fn

class PreCachedAMDDataset(Dataset):
    """
    Wraps AMDForexDataset and pre-caches ALL samples as contiguous tensors.
    __getitem__ becomes a pure tensor index — zero pandas overhead per batch.
    """

    def __init__(self, amd_dataset: AMDForexDataset):
        print(f"    Pre-caching {len(amd_dataset):,} samples...", end=" ", flush=True)
        t0 = time.time()

        sample = amd_dataset[0]
        skip_keys = {"label", "phase_label"}
        self.timeframes = [k for k in sample if k not in skip_keys]
        self.feat_keys = {tf: list(sample[tf].keys()) for tf in self.timeframes}
        self.has_phase = "phase_label" in sample

        n = len(amd_dataset)
        self.tensors = {}
        for tf in self.timeframes:
            self.tensors[tf] = {}
            for feat in self.feat_keys[tf]:
                shape = sample[tf][feat].shape
                self.tensors[tf][feat] = torch.empty(n, *shape, dtype=torch.float32)
        self.labels = torch.empty(n, dtype=torch.long)
        self.phase_labels = torch.empty(n, dtype=torch.long) if self.has_phase else None

        for i in range(n):
            item = amd_dataset[i]
            for tf in self.timeframes:
                for feat in self.feat_keys[tf]:
                    self.tensors[tf][feat][i] = item[tf][feat]
            self.labels[i] = item["label"]
            if self.has_phase:
                self.phase_labels[i] = item["phase_label"]

        elapsed = time.time() - t0
        print(f"done in {elapsed:.0f}s")

    def __len__(self):
        return self.labels.shape[0]

    def __getitem__(self, idx):
        result = {}
        for tf in self.timeframes:
            result[tf] = {feat: self.tensors[tf][feat][idx] for feat in self.feat_keys[tf]}
        result["label"] = self.labels[idx]
        if self.phase_labels is not None:
            result["phase_label"] = self.phase_labels[idx]
        return result


def build_combined_amd_dataset(split_data, split_name, config, pairs, cache=True):
    """Build one AMDForexDataset per pair, then concatenate."""
    datasets = []
    for pair_tag in pairs:
        if pair_tag not in split_data:
            continue
        pair_dfs = split_data[pair_tag][split_name]
        # Skip if 5min data is too short
        if len(pair_dfs.get("5min", pd.DataFrame())) < config.lookbacks["5min"] + LOOKAHEAD + 10:
            print(f"  {pair_tag}: too few 5min bars — skipping")
            continue

        ds = AMDForexDataset(
            dataframes=pair_dfs,
            config=config,
            lookahead=LOOKAHEAD,
            threshold_pips=THRESHOLD_PIPS,
            pair=PAIR_NAMES[pair_tag],
        )
        if cache and len(ds) > 0:
            ds = PreCachedAMDDataset(ds)
        datasets.append(ds)
        print(f"  {pair_tag}: {len(ds):,} samples")

    if not datasets:
        raise ValueError(f"No datasets built for {split_name}!")
    combined = ConcatDataset(datasets)
    print(f"  Total: {len(combined):,} samples")
    return combined


print("Building & caching TRAIN dataset:")
train_dataset = build_combined_amd_dataset(split_data, "train", config, PAIRS, cache=True)

print("\nBuilding & caching VAL dataset:")
val_dataset = build_combined_amd_dataset(split_data, "val", config, PAIRS, cache=True)

print("\nBuilding & caching TEST dataset:")
test_dataset = build_combined_amd_dataset(split_data, "test", config, PAIRS, cache=True)

# DataLoaders — GPU-optimized settings
NUM_WORKERS = min(4, os.cpu_count() or 1)
PREFETCH = 4

train_loader = DataLoader(
    train_dataset,
    batch_size=config.batch_size,
    shuffle=True,
    num_workers=NUM_WORKERS,
    collate_fn=amd_collate_fn,
    pin_memory=(device.type == "cuda"),
    persistent_workers=(NUM_WORKERS > 0),
    prefetch_factor=PREFETCH if NUM_WORKERS > 0 else None,
    drop_last=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config.batch_size * 2,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=amd_collate_fn,
    pin_memory=(device.type == "cuda"),
    persistent_workers=(NUM_WORKERS > 0),
    prefetch_factor=PREFETCH if NUM_WORKERS > 0 else None,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=config.batch_size * 2,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=amd_collate_fn,
    pin_memory=(device.type == "cuda"),
    persistent_workers=(NUM_WORKERS > 0),
    prefetch_factor=PREFETCH if NUM_WORKERS > 0 else None,
)

print(f"\nDataLoaders ready:")
print(f"  Train: {len(train_loader):,} batches × {config.batch_size}")
print(f"  Val:   {len(val_loader):,} batches × {config.batch_size * 2}")
print(f"  Test:  {len(test_loader):,} batches")
print(f"  Workers: {NUM_WORKERS}, pin_memory={device.type == 'cuda'}, prefetch={PREFETCH}")

## 7. Model Architecture

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 7: Instantiate model + torch.compile for kernel fusion
# ═══════════════════════════════════════════════════════════════
model = AMDScalper(config)
model = model.to(device)

n_params = model.count_parameters()
print(f"AMD Scalper Model")
print(f"{'='*50}")
print(f"Total parameters   : {n_params:,}")
print(f"Timeframes         : {config.timeframes}")
print(f"Entry TF           : {config.entry_timeframe}")
print(f"TF wave dim        : {config.tf_wave_dim}")
print(f"Fused dim          : {config.fused_dim}")
print(f"Predictor layers   : {config.predictor_layers}")
print(f"Predictor heads    : {config.predictor_heads}")
print(f"Entry models       : {config.n_entry_models} (S&R, S&D, ORB)")
print(f"Phase classes      : {config.n_phase_classes} (ACCUM/MANIP/DIST/INVALID)")
print(f"Signal classes     : {config.n_signal_classes} (BUY/SELL/HOLD)")
print(f"Device             : {device}")

# torch.compile — fuses kernels, eliminates Python overhead (15-30% speedup)
if device.type == "cuda" and hasattr(torch, "compile"):
    try:
        model = torch.compile(model, mode="reduce-overhead")
        print(f"torch.compile     : enabled (reduce-overhead mode)")
    except Exception as e:
        print(f"torch.compile     : skipped ({e})")
else:
    print(f"torch.compile     : disabled (CPU or old PyTorch)")

print()

# Quick forward pass sanity check (also triggers compile warmup)
sample_batch = next(iter(train_loader))
model_input = {
    tf: {k: v.to(device) for k, v in sample_batch[tf].items()}
    for tf in config.timeframes
    if tf in sample_batch and isinstance(sample_batch[tf], dict)
}
with torch.no_grad():
    out = model(model_input)

print("Output tensors:")
for k, v in out.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k:20s}: {list(v.shape)}")
print("\nForward pass OK — gate weight distribution:")
gw = out["gate_weights"][0].detach().cpu()
print(f"  S&R: {gw[0]:.3f}, S&D: {gw[1]:.3f}, ORB: {gw[2]:.3f}")

## 8. Training Setup — Optimizer, Loss, Scheduler

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 8: Loss, optimizer, scheduler, mixed precision
# ═══════════════════════════════════════════════════════════════
from wavetrader.train_amd_scalper import AMDScalperLoss

criterion = AMDScalperLoss(
    signal_weight=1.0,        # Main BUY/SELL/HOLD loss
    phase_weight=0.3,         # AMD phase classification
    entry_aux_weight=0.15,    # Per-entry-model auxiliary losses (S&R, S&D, ORB)
    gate_entropy_weight=0.05, # Encourage using all 3 entry models
    conf_weight=0.1,          # Confidence calibration
    label_smoothing=0.1,
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config.learning_rate,
    weight_decay=0.01,
    betas=(0.9, 0.999),
    fused=(device.type == "cuda"),  # Fused AdamW — fewer kernel launches
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=config.epochs,
    eta_min=config.learning_rate * 0.01,
)

# Mixed precision — non-deprecated API
use_amp = device.type == "cuda"
scaler = GradScaler("cuda", enabled=use_amp)

print(f"Optimizer     : AdamW (lr={config.learning_rate}, wd=0.01, fused={device.type == 'cuda'})")
print(f"Scheduler     : CosineAnnealing (T_max={config.epochs})")
print(f"Mixed precision: {'Enabled (float16 + TF32)' if use_amp else 'Disabled (CPU mode)'}")
print(f"Loss weights  : signal=1.0, phase=0.3, entry_aux=0.15, gate_ent=0.05, conf=0.1")

## 9. Training Loop — GPU Accelerated with Mixed Precision

AMD Scalper has extra outputs vs WaveFollower: phase logits, 3 entry model logits, and gate weights. The training loop passes phase labels and tracks per-component losses.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 9: Full training loop — GPU-optimized with AMD-specific losses
# ═══════════════════════════════════════════════════════════════

def train_one_epoch(model, loader, optimizer, criterion, scaler, device, config, use_amp):
    model.train()
    total_loss = 0.0
    correct, n_samples = 0, 0
    loss_components = {
        "signal": 0, "phase": 0, "entry_aux": 0, "gate_ent": 0, "conf": 0,
    }

    for batch in loader:
        signal_labels = batch["label"].to(device, non_blocking=True)
        phase_labels = batch.get("phase_label")
        if phase_labels is not None:
            phase_labels = phase_labels.to(device, non_blocking=True)

        model_input = {
            tf: {k: v.to(device, non_blocking=True) for k, v in batch[tf].items()}
            for tf in config.timeframes
            if tf in batch and isinstance(batch[tf], dict)
        }

        optimizer.zero_grad(set_to_none=True)

        with autocast("cuda", enabled=use_amp):
            out = model(model_input)
            losses = criterion(out, signal_labels, phase_labels)

        scaler.scale(losses["total"]).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += losses["total"].item()
        correct += (out["signal_logits"].argmax(-1) == signal_labels).sum().item()
        n_samples += signal_labels.size(0)

        loss_components["signal"] += losses["signal_loss"].item()
        loss_components["phase"] += losses["phase_loss"].item()
        loss_components["entry_aux"] += losses["entry_aux_loss"].item()
        loss_components["gate_ent"] += losses["gate_entropy"].item()
        loss_components["conf"] += losses["conf_loss"].item()

    n_batches = max(len(loader), 1)
    return {
        "loss": total_loss / n_batches,
        "accuracy": correct / max(n_samples, 1),
        "signal_loss": loss_components["signal"] / n_batches,
        "phase_loss": loss_components["phase"] / n_batches,
        "entry_aux_loss": loss_components["entry_aux"] / n_batches,
        "gate_entropy": loss_components["gate_ent"] / n_batches,
        "conf_loss": loss_components["conf"] / n_batches,
    }


@torch.no_grad()
def validate(model, loader, criterion, device, config, use_amp):
    model.eval()
    total_loss = 0.0
    correct, n_samples = 0, 0
    all_preds, all_labels = [], []
    gate_sums = torch.zeros(3)
    gate_count = 0

    for batch in loader:
        signal_labels = batch["label"].to(device, non_blocking=True)
        phase_labels = batch.get("phase_label")
        if phase_labels is not None:
            phase_labels = phase_labels.to(device, non_blocking=True)

        model_input = {
            tf: {k: v.to(device, non_blocking=True) for k, v in batch[tf].items()}
            for tf in config.timeframes
            if tf in batch and isinstance(batch[tf], dict)
        }

        with autocast("cuda", enabled=use_amp):
            out = model(model_input)
            losses = criterion(out, signal_labels, phase_labels)

        preds = out["signal_logits"].argmax(-1)
        total_loss += losses["total"].item()
        correct += (preds == signal_labels).sum().item()
        n_samples += signal_labels.size(0)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(signal_labels.cpu().tolist())

        # Track gate usage
        gate_sums += out["gate_weights"].mean(0).cpu()
        gate_count += 1

    n_batches = max(len(loader), 1)
    avg_gate = gate_sums / max(gate_count, 1)
    return {
        "loss": total_loss / n_batches,
        "accuracy": correct / max(n_samples, 1),
        "predictions": all_preds,
        "labels": all_labels,
        "gate_avg": avg_gate.tolist(),
    }


# ── Main training loop ──────────────────────────────────────
history = {
    "train_loss": [], "val_loss": [],
    "train_acc": [], "val_acc": [],
    "lr": [],
    "signal_loss": [], "phase_loss": [],
    "gate_sr": [], "gate_sd": [], "gate_orb": [],
}
best_val_acc = 0.0
best_epoch = 0
checkpoint_path = CHECKPOINT_DIR / "model_weights.pt"

print(f"Training AMD Scalper for {config.epochs} epochs")
print(f"{'='*80}")
print(f"{'Ep':>3} | {'TrLoss':>7} | {'VlLoss':>7} | {'TrAcc':>6} | {'VlAcc':>6} | "
      f"{'Sig':>5} | {'Phs':>5} | {'Gate S/D/O':>12} | {'LR':>9} | {'T':>4}")
print(f"{'-'*80}")

training_start = time.time()

for epoch in range(config.epochs):
    epoch_start = time.time()

    # Train
    train_metrics = train_one_epoch(
        model, train_loader, optimizer, criterion, scaler, device, config, use_amp
    )

    # Validate
    val_metrics = validate(model, val_loader, criterion, device, config, use_amp)

    # Step scheduler
    scheduler.step()
    current_lr = optimizer.param_groups[0]["lr"]

    # Record history
    history["train_loss"].append(train_metrics["loss"])
    history["val_loss"].append(val_metrics["loss"])
    history["train_acc"].append(train_metrics["accuracy"])
    history["val_acc"].append(val_metrics["accuracy"])
    history["lr"].append(current_lr)
    history["signal_loss"].append(train_metrics["signal_loss"])
    history["phase_loss"].append(train_metrics["phase_loss"])
    g = val_metrics["gate_avg"]
    history["gate_sr"].append(g[0])
    history["gate_sd"].append(g[1])
    history["gate_orb"].append(g[2])

    elapsed = time.time() - epoch_start
    star = ""

    # Save best checkpoint
    if val_metrics["accuracy"] > best_val_acc:
        best_val_acc = val_metrics["accuracy"]
        best_epoch = epoch + 1
        star = " *"
        torch.save({
            "model_state_dict": model.state_dict(),
            "config": {
                "model_type": "amd_scalper",
                "timeframes": config.timeframes,
                "lookbacks": config.lookbacks,
                "entry_timeframe": config.entry_timeframe,
                "tf_wave_dim": config.tf_wave_dim,
                "fused_dim": config.fused_dim,
                "predictor_layers": config.predictor_layers,
                "predictor_heads": config.predictor_heads,
                "n_entry_models": config.n_entry_models,
                "n_signal_classes": config.n_signal_classes,
                "n_phase_classes": config.n_phase_classes,
                "structure_emb_dim": config.structure_emb_dim,
                "price_conv_dim": config.price_conv_dim,
                "entry_head_dim": config.entry_head_dim,
                "predictor_ff_dim": config.predictor_ff_dim,
                "predictor_hidden": config.predictor_hidden,
                "dropout": config.dropout,
            },
            "epoch": epoch + 1,
            "val_accuracy": val_metrics["accuracy"],
            "val_loss": val_metrics["loss"],
            "optimizer_state_dict": optimizer.state_dict(),
        }, str(checkpoint_path))

    print(
        f"{epoch+1:3d} | {train_metrics['loss']:7.4f} | {val_metrics['loss']:7.4f} | "
        f"{train_metrics['accuracy']:5.2%} | {val_metrics['accuracy']:5.2%} | "
        f"{train_metrics['signal_loss']:5.3f} | {train_metrics['phase_loss']:5.3f} | "
        f"{g[0]:.2f}/{g[1]:.2f}/{g[2]:.2f} | "
        f"{current_lr:9.2e} | {elapsed:3.0f}s{star}"
    )

total_time = time.time() - training_start
print(f"{'='*80}")
print(f"Training complete in {total_time/60:.1f} min")
print(f"Best val accuracy: {best_val_acc:.2%} at epoch {best_epoch}")
print(f"Checkpoint: {checkpoint_path}")

## 10. Training Curves & Gate Analysis

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 10: Plot training curves + gate weight evolution
# ═══════════════════════════════════════════════════════════════
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=(
        "Loss", "Accuracy",
        "Component Losses (Signal vs Phase)", "Learning Rate",
        "Gate Weights (S&R / S&D / ORB)", "Train vs Val Gap",
    ),
    vertical_spacing=0.08,
)

epochs_x = list(range(1, len(history["train_loss"]) + 1))

# Loss
fig.add_trace(go.Scatter(x=epochs_x, y=history["train_loss"], name="Train Loss",
              line=dict(color="#f85149")), row=1, col=1)
fig.add_trace(go.Scatter(x=epochs_x, y=history["val_loss"], name="Val Loss",
              line=dict(color="#58a6ff")), row=1, col=1)

# Accuracy
fig.add_trace(go.Scatter(x=epochs_x, y=history["train_acc"], name="Train Acc",
              line=dict(color="#3fb950")), row=1, col=2)
fig.add_trace(go.Scatter(x=epochs_x, y=history["val_acc"], name="Val Acc",
              line=dict(color="#bc8cff")), row=1, col=2)

# Component losses
fig.add_trace(go.Scatter(x=epochs_x, y=history["signal_loss"], name="Signal Loss",
              line=dict(color="#f85149")), row=2, col=1)
fig.add_trace(go.Scatter(x=epochs_x, y=history["phase_loss"], name="Phase Loss",
              line=dict(color="#58a6ff")), row=2, col=1)

# Learning Rate
fig.add_trace(go.Scatter(x=epochs_x, y=history["lr"], name="LR",
              line=dict(color="#d29922")), row=2, col=2)

# Gate weights — should converge to roughly equal usage (entropy regularised)
fig.add_trace(go.Scatter(x=epochs_x, y=history["gate_sr"], name="S&R Gate",
              line=dict(color="#f85149")), row=3, col=1)
fig.add_trace(go.Scatter(x=epochs_x, y=history["gate_sd"], name="S&D Gate",
              line=dict(color="#3fb950")), row=3, col=1)
fig.add_trace(go.Scatter(x=epochs_x, y=history["gate_orb"], name="ORB Gate",
              line=dict(color="#58a6ff")), row=3, col=1)
fig.add_hline(y=1/3, line_dash="dash", line_color="gray", row=3, col=1)

# Overfit gap
gap = [t - v for t, v in zip(history["train_acc"], history["val_acc"])]
fig.add_trace(go.Scatter(x=epochs_x, y=gap, name="Acc Gap",
              fill="tozeroy", line=dict(color="#f85149")), row=3, col=2)
fig.add_hline(y=0.05, line_dash="dash", line_color="gray", row=3, col=2)

fig.update_layout(
    height=900, width=1000,
    title_text="AMD Scalper Training Curves",
    template="plotly_dark",
    paper_bgcolor="#0d1117",
    plot_bgcolor="#161b22",
    showlegend=False,
)
fig.show()

## 11. Test Set Evaluation & Per-Class Breakdown

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 11: Test evaluation with confusion matrix + phase analysis
# ═══════════════════════════════════════════════════════════════

# Reload best checkpoint
best_state = torch.load(str(checkpoint_path), weights_only=False, map_location=device)
model.load_state_dict(best_state["model_state_dict"])
print(f"Loaded best checkpoint (epoch {best_state['epoch']}, val_acc={best_state['val_accuracy']:.2%})")

# Evaluate on test set
test_metrics = validate(model, test_loader, criterion, device, config, use_amp)

print(f"\nTest Results:")
print(f"  Loss     : {test_metrics['loss']:.4f}")
print(f"  Accuracy : {test_metrics['accuracy']:.2%}")
print(f"  Gate avg : S&R={test_metrics['gate_avg'][0]:.3f}, "
      f"S&D={test_metrics['gate_avg'][1]:.3f}, ORB={test_metrics['gate_avg'][2]:.3f}")

# Per-class breakdown
from collections import Counter
SIGNAL_NAMES = ["BUY", "SELL", "HOLD"]

preds = test_metrics["predictions"]
labels = test_metrics["labels"]

print(f"\nClass Distribution:")
label_counts = Counter(labels)
pred_counts = Counter(preds)
for i, name in enumerate(SIGNAL_NAMES):
    n_true = label_counts.get(i, 0)
    n_pred = pred_counts.get(i, 0)
    class_correct = sum(1 for p, l in zip(preds, labels) if l == i and p == i)
    class_total = label_counts.get(i, 1)
    class_acc = class_correct / max(class_total, 1)
    print(f"  {name:5s}: true={n_true:5d}, pred={n_pred:5d}, acc={class_acc:.2%}")

# Confusion matrix
print(f"\nConfusion Matrix:")
print(f"{'':>10} {'BUY':>8} {'SELL':>8} {'HOLD':>8}")
for i, name in enumerate(SIGNAL_NAMES):
    row = [sum(1 for p, l in zip(preds, labels) if l == i and p == j) for j in range(3)]
    print(f"{name:>10} {row[0]:8d} {row[1]:8d} {row[2]:8d}")

# BUY/SELL-only accuracy (excluding HOLD)
trade_preds = [(p, l) for p, l in zip(preds, labels) if l in (0, 1)]
if trade_preds:
    trade_acc = sum(1 for p, l in trade_preds if p == l) / len(trade_preds)
    print(f"\nBUY/SELL accuracy (excluding HOLD): {trade_acc:.2%} ({len(trade_preds)} samples)")

## 12. Save Final Checkpoint & Summary

Saves the model in the format expected by the dashboard and streaming engine. The checkpoint is saved to `checkpoints/amd_scalper/model_weights.pt` — the same path referenced in `docker-compose.yml`.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 12: Save final checkpoint + training metadata
# ═══════════════════════════════════════════════════════════════

final_checkpoint = {
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "config": {
        "model_type": "amd_scalper",
        "timeframes": config.timeframes,
        "lookbacks": config.lookbacks,
        "entry_timeframe": config.entry_timeframe,
        "tf_wave_dim": config.tf_wave_dim,
        "fused_dim": config.fused_dim,
        "predictor_layers": config.predictor_layers,
        "predictor_heads": config.predictor_heads,
        "predictor_hidden": config.predictor_hidden,
        "predictor_ff_dim": config.predictor_ff_dim,
        "n_entry_models": config.n_entry_models,
        "n_signal_classes": config.n_signal_classes,
        "n_phase_classes": config.n_phase_classes,
        "structure_emb_dim": config.structure_emb_dim,
        "price_conv_dim": config.price_conv_dim,
        "entry_head_dim": config.entry_head_dim,
        "dropout": config.dropout,
        "pair": "GBP/JPY",
    },
    "training": {
        "epochs": config.epochs,
        "learning_rate": config.learning_rate,
        "batch_size": config.batch_size,
        "best_epoch": best_epoch,
        "best_val_accuracy": best_val_acc,
        "test_accuracy": test_metrics["accuracy"],
        "test_loss": test_metrics["loss"],
        "pairs_trained_on": list(PAIR_NAMES.values()),
        "total_train_samples": len(train_dataset),
        "total_val_samples": len(val_dataset),
        "total_test_samples": len(test_dataset),
        "training_time_min": total_time / 60,
        "lookahead": LOOKAHEAD,
        "threshold_pips": THRESHOLD_PIPS,
        "gate_avg": test_metrics["gate_avg"],
        "timestamp": datetime.now().isoformat(),
        "device": str(device),
    },
    "history": history,
}

# Save checkpoint
torch.save(final_checkpoint, str(checkpoint_path))
print(f"Checkpoint saved: {checkpoint_path}")
print(f"  Size: {checkpoint_path.stat().st_size / 1e6:.1f} MB")

# Timestamped copy
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
timestamped_path = CHECKPOINT_DIR / f"amd_scalper_{ts}.pt"
torch.save(final_checkpoint, str(timestamped_path))
print(f"Timestamped copy: {timestamped_path}")

# Training history as JSON
history_path = CHECKPOINT_DIR / "training_history.json"
with open(history_path, "w") as f:
    json.dump(history, f, indent=2)
print(f"History saved: {history_path}")

print(f"\n{'='*50}")
print(f"AMD Scalper Training Complete")
print(f"{'='*50}")
print(f"  Parameters     : {model.count_parameters():,}")
print(f"  Best val acc   : {best_val_acc:.2%} (epoch {best_epoch})")
print(f"  Test accuracy  : {test_metrics['accuracy']:.2%}")
print(f"  Training time  : {total_time/60:.1f} min")
print(f"  Pairs trained  : {', '.join(PAIR_NAMES.values())}")
print(f"  Gate balance   : S&R={test_metrics['gate_avg'][0]:.3f}, "
      f"S&D={test_metrics['gate_avg'][1]:.3f}, ORB={test_metrics['gate_avg'][2]:.3f}")
print(f"  OANDA account  : 101-001-30902818-003")
print(f"\nNext steps:")
print(f"  1. Run AMDScalper_Validation.ipynb for holdout eval + backtest")
print(f"  2. Create OANDA sub-account and regenerate API key")
print(f"  3. Uncomment wavetrader-amd-scalper in docker-compose.yml")
print(f"  4. docker compose up -d --build")
print(f"  5. Select 'AMD Scalper' in the dashboard dropdown")

## 13. Save to Google Drive

Copy the AMD Scalper checkpoint to Google Drive alongside MTF and WaveFollower checkpoints.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 13: Save AMD Scalper checkpoint to Google Drive
# ═══════════════════════════════════════════════════════════════
import shutil

# ── Mount Google Drive (skip if already mounted) ─────────────────────────────
DRIVE_MOUNT = Path("/content/drive")
if DRIVE_MOUNT.exists():
    print("Google Drive already mounted.")
else:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except ImportError:
        print("Not running on Colab — mount Drive manually or use Drive for Desktop.")
        print("Falling back to local save only.")

DRIVE_ROOT = Path("/content/drive/MyDrive/phase_lm")
DRIVE_CKPT = DRIVE_ROOT / "checkpoints"

if DRIVE_ROOT.exists():
    # ── Create timestamped directory ──────────────────────────────────────────
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    SAVE_DIR = DRIVE_CKPT / f"amd_scalper_{ts}"
    SAVE_DIR.mkdir(parents=True, exist_ok=True)

    # 1. Copy model weights
    drive_weights = SAVE_DIR / "model_weights.pt"
    shutil.copy(str(checkpoint_path), str(drive_weights))

    # 2. Save config as JSON
    cfg_dict = {
        "model_type": "amd_scalper",
        "timeframes": config.timeframes,
        "lookbacks": config.lookbacks,
        "entry_timeframe": config.entry_timeframe,
        "tf_wave_dim": config.tf_wave_dim,
        "fused_dim": config.fused_dim,
        "predictor_layers": config.predictor_layers,
        "predictor_heads": config.predictor_heads,
        "predictor_hidden": config.predictor_hidden,
        "predictor_ff_dim": config.predictor_ff_dim,
        "n_entry_models": config.n_entry_models,
        "n_signal_classes": config.n_signal_classes,
        "n_phase_classes": config.n_phase_classes,
        "structure_emb_dim": config.structure_emb_dim,
        "price_conv_dim": config.price_conv_dim,
        "entry_head_dim": config.entry_head_dim,
        "dropout": config.dropout,
        "pair": "GBP/JPY",
        "_run": ts,
        "_mode": "amd_scalper",
        "_arch_class": "AMDScalper",
        "_best_val_acc": float(best_val_acc),
        "_test_acc": float(test_metrics["accuracy"]),
        "epochs": config.epochs,
        "learning_rate": config.learning_rate,
        "batch_size": config.batch_size,
        "lookahead": LOOKAHEAD,
        "threshold_pips": THRESHOLD_PIPS,
    }
    with open(SAVE_DIR / "config.json", "w") as f:
        json.dump(cfg_dict, f, indent=2, default=str)

    # 3. Save training history
    with open(SAVE_DIR / "history.json", "w") as f:
        json.dump(history, f, indent=2)

    # 4. Compute SHA-256 for integrity verification
    def _sha256(path):
        h = hashlib.sha256()
        with open(path, "rb") as fh:
            for chunk in iter(lambda: fh.read(65536), b""):
                h.update(chunk)
        return h.hexdigest()

    weights_hash = _sha256(drive_weights)
    with open(SAVE_DIR / "weights.sha256", "w") as f:
        f.write(f"{weights_hash}  model_weights.pt\n")

    # ── Summary ───────────────────────────────────────────────────────────────
    print(f"\nSaved to Google Drive:")
    for item in sorted(SAVE_DIR.iterdir()):
        size_kb = item.stat().st_size / 1024
        print(f"  {item.name:<35} {size_kb:>8.1f} KB")
    print(f"\nSHA-256: {weights_hash[:32]}…")
    print(f"Best val acc : {best_val_acc:.2%}")
    print(f"Test acc     : {test_metrics['accuracy']:.2%}")
    print(f"\nFull path: {SAVE_DIR}")
else:
    print("Google Drive not available at /content/drive/MyDrive/phase_lm")
    print(f"Local checkpoint is at: {checkpoint_path}")